<a href="https://colab.research.google.com/github/munawirmt/English_to_Malayalam_Translation_Seq2Seq/blob/main/English_to_Malayalam_Translation_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ENCODER AND DECODER IN NLP**

HERE IAM GOING TO ENCODE AND DECODE FROM ENGLISH TO MALAYALAM

*MODEL ARCHITECTURE:*


> Encoder (Input -> Embedding -> LSTM) -> Context Vector -> Decoder (Input -> Embedding -> LSTM -> Dense) -> Output



In [3]:
## 1. LIBRERY INSTALLATION
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Embedding, Input, LSTM
from tensorflow.keras.models import Model

**PARAMETERS**

*   SIZE OF VOCABILARY IN ENGLISH & MALAYALAM
*   DIMENSION OF EMBEDDING LAYERS
*   MEMMORY SIZE OF LSTM (UNITS)



In [4]:
##2. PARAMETERS
vocab_size_en = 10000
vocab_size_ml = 5000
embedding_dim = 128
units = 256

**ENCODER**

*   INPUT WORD ("I AM COMING")
*   GIVING CONTEXT VECTOR
*   TRANSFERING LAST TWO STATES(h,c) TO DECODER


In [5]:
##3. ENCODER
encoder_inputs = Input(shape=(None,), name="encoder_input")
enc_emb = Embedding(vocab_size_en, embedding_dim, name="encoder_embedding")(
    encoder_inputs
)
encoder_lstm = LSTM(units, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]


**DECODER**

*   DECODER INPUT ( < start > I AM COMING )
*   TAKE encoder_states AS intial_state OF DECODER LSTM
*   PREDICT EACH WORDS



In [6]:
##4. DECODER
decoder_inputs = Input(shape=(None,), name="decoder_input")
decoder_embedding_layer = Embedding(
    vocab_size_ml, embedding_dim, name="decoder_embedding"
)
dec_emb = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    units, return_sequences=True, return_state=True, name="decoder_lstm"
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_ml, activation='softmax', name="decoder_output")
decoder_outputs = decoder_dense(decoder_outputs)

**FINAL MODEL**


In [8]:
##5. MODEL
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 128) │  1,280,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 128) │    640,000 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    394,240 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, None,      │  1,285,000 │ decoder_lstm[0][… │
│ (Dense)             │ 5000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,993,480 (15.23 MB)

 Trainable params: 3,993,480 (15.23 MB)

 Non-trainable params: 0 (0.00 B)

**TRAINING**

*   ENCODER INPUT TEXT: "I am reading"
*   DECODER INPUT TEXT: "< start > ഞാൻ വായിക്കുന്നു"
*   DECODER OUTPUT TEXT: "ഞാൻ വായിക്കുന്നു < end >"

In [9]:
##6. TRAINING
encoder_input_data = [[5, 23, 45]]
decoder_input_data = [[1, 67, 89, 12]]
decoder_target_data = [[67, 89, 12, 2]]

encoder_input_arr = np.array(encoder_input_data)
decoder_input_arr = np.array(decoder_input_data)
decoder_target_arr = np.array(decoder_target_data)

model.fit([encoder_input_arr, decoder_input_arr], decoder_target_arr, epochs=200)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 8.5175
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 8.5081
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 8.4979
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 8.4861
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 8.4715
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 8.4528
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 8.4279
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 8.3943
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - loss: 8.3486
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 8.2859
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 8.1991
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 8.0783
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 7.9095
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 7.6736
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 7.3479
Epoch 16/200
1/1 ━━━

**SPLIT MODEL TO 2 WHEN REAL USE FOR INFERENCE**


*   ENCODER MODEL
*   DECODER MODEL

In [10]:
## 7. INFERENCE
 # A. ENCODER MODEL
encoder_model = Model(encoder_inputs, encoder_states)

In [11]:
# B. DECODER MODEL
decoder_state_input_h = Input(shape=(units,))
decoder_state_input_c = Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# --- തിരുത്തൽ: ട്രെയിൻ ചെയ്ത ലെയർ ഇവിടെ വീണ്ടും ഉപയോഗിക്കുന്നു ---
dec_emb2 = decoder_embedding_layer(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2
)


**TEST WORD BY WORD**

In [ ]:
#token id of iam coming
test_input = np.array([[5, 23, 90]])

states_value = encoder_model.predict(test_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


In [ ]:
## find forst word
target_seq = np.array([[1]])

output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

next_word_id = np.argmax(output_tokens[0, -1, :])
print("ID of first word:", next_word_id)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
ID of first word: 89


In [ ]:
## find second word
target_seq = np.array([[next_word_id]])

states_value = [h, c]

output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

next_word_id = np.argmax(output_tokens[0, -1, :])
print("രണ്ടാമത്തെ വാക്കിന്റെ ID:", next_word_id)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
രണ്ടാമത്തെ വാക്കിന്റെ ID: 89


In [ ]:
## find third word
target_seq = np.array([[next_word_id]])
states_value = [h, c]

output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
next_word_id = np.argmax(output_tokens[0, -1, :])
print("മൂന്നാമത്തെ വാക്കിന്റെ ID:", next_word_id)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
മൂന്നാമത്തെ വാക്കിന്റെ ID: 2


**TO GET FULL DECODING IN ONE STEP WE USING LOOP**

In [12]:
def decode_sequence(input_seq):
  states_value = encoder_model.predict(input_seq)
  target_seq = np.zeros((1, 1))
  target_seq[0, 0] = 1  # <start> token ID

  stop_condition = False
  decoded_sentence = ''

  while not stop_condition:
    output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
    sampled_token_index = np.argmax(output_tokens[0, -1, :])
    sampled_word = reverse_target_word_index.get(sampled_token_index, '')

    if sampled_word == '<end>' or len(decoded_sentence.split()) > 20:
      stop_condition = True
    else:
      decoded_sentence += ' ' + sampled_word
      target_seq = np.zeros((1, 1))
      target_seq[0, 0] = sampled_token_index
      states_value = [h, c]

  return decoded_sentence.strip()

**ID TO WORD DICTIONARY MAPPING**

In [13]:
## Define the ID-to-word dictionary mapping for Malayalam
reverse_target_word_index = {
    1: '<start>',
    67: 'ഞാൻ',
    89: 'വായിക്കുന്നു',
    12: 'വരുന്നു',  # ഒഴിഞ്ഞ സ്പേസിന് പകരം വാക്ക് നൽകി
    2: '<end>',
}

**TEST TRANSLATION BY DECODING**

In [14]:
## RUN LOOP AND TEST
test_input = np.array([[5, 23, 45]])
predicted_malayalam = decode_sequence(test_input)
print("Malayalam Translation:", predicted_malayalam)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Malayalam Translation: ഞാൻ വായിക്കുന്നു വരുന്നു


# **USING KERAS TOKENIZER TO AUTOMATIC WORD ENCODING TO NUMBERS**

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [16]:
# 1. MAKING DATA LIST
english_sentences = ["I am reading", "I am coming", "He is going"]
malayalam_sentences = ["<start> ഞാൻ വായിക്കുന്നു <end>", "<start> ഞാൻ വരുന്നു <end>", "<start> അവൻ പോകുന്നു <end>"]

**ENGLISH**

In [17]:
# 2. SETTING ENGLISH TOKENIZER
text_tokenizer = Tokenizer()
text_tokenizer.fit_on_texts(english_sentences)

In [18]:
## 3. MAKING TOKENS TO NUMBERS
encoder_input_sequences = text_tokenizer.texts_to_sequences(english_sentences)

In [19]:
## 4.PADDING
encoder_input_data = pad_sequences(encoder_input_sequences, padding='post')

**MALAYALAM**

In [20]:
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [21]:
malayalam_sentences = [
    "<start> ഞാൻ വായിക്കുന്നു <end>",
    "<start> ഞാൻ വരുന്നു <end>"
]

In [22]:
ml_tokenizer = Tokenizer(filters='') # ചിഹ്നങ്ങൾ ഒഴിവാക്കാതിരിക്കാൻ filters='' നൽകാം
ml_tokenizer.fit_on_texts(malayalam_sentences)

**SAVE TOKENIZER AND MODEL**

In [23]:
## SAVE TOKENIZER
# 1. MODEL SAVE
model.save("english_to_malayalam_model.keras")
encoder_model.save("encoder_model.keras")
decoder_model.save("decoder_model.keras")

# 2. TOKENIZER SAVE
with open('malayalam_tokenizer.pickle', 'wb') as handle:
    pickle.dump(ml_tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Models and Tokenizer saved successfully!")


Models and Tokenizer saved successfully!


**LOAD TOKENIZER AND MODEL**

In [24]:
import pickle
from tensorflow.keras.models import load_model

In [25]:
# 1. LOAD MODEL
encoder_model = load_model("encoder_model.keras")
decoder_model = load_model("decoder_model.keras")

# 2. LOAD TOKENIZER
with open('malayalam_tokenizer.pickle', 'rb') as handle:
    ml_tokenizer = pickle.load(handle)

In [26]:
# 3. MAKING reverse_target_word_index AUTOMATICALLY FROM TOKENIZER
reverse_target_word_index = ml_tokenizer.index_word

In [27]:
# 4. FIND ID OF TOKENIZER AUTOMATICALLY
start_token_id = ml_tokenizer.word_index['<start>']

# **FINAL CODE OF TRANSLATION PROJECT**

In [28]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

english_sentences = ["I am reading", "I am coming"]
malayalam_input_sentences = ["<start> ഞാൻ വായിക്കുന്നു", "<start> ഞാൻ വരുന്നു"]
malayalam_target_sentences = ["ഞാൻ വായിക്കുന്നു <end>", "ഞാൻ വരുന്നു <end>"]

latent_dim = 256
embedding_dim = 128

en_tokenizer = Tokenizer()
en_tokenizer.fit_on_texts(english_sentences)
encoder_input_data = en_tokenizer.texts_to_sequences(english_sentences)
encoder_input_data = pad_sequences(encoder_input_data, padding='post')
vocab_size_en = len(en_tokenizer.word_index) + 1

ml_tokenizer = Tokenizer(filters='')
ml_tokenizer.fit_on_texts(malayalam_input_sentences + malayalam_target_sentences)
vocab_size_ml = len(ml_tokenizer.word_index) + 1

decoder_input_sequences = ml_tokenizer.texts_to_sequences(malayalam_input_sentences)
decoder_input_data = pad_sequences(decoder_input_sequences, padding='post')

decoder_target_sequences = ml_tokenizer.texts_to_sequences(malayalam_target_sentences)
decoder_target_data = pad_sequences(decoder_target_sequences, padding='post')

encoder_inputs = Input(shape=(None,), name="encoder_input")
enc_emb = Embedding(vocab_size_en, embedding_dim, name="encoder_embedding")(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,), name="decoder_input")
decoder_embedding_layer = Embedding(vocab_size_ml, embedding_dim, name="decoder_embedding")
dec_emb = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_ml, activation='softmax', name="decoder_output")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit([encoder_input_data, decoder_input_data], decoder_target_data, epochs=200, verbose=0)
print("Training Completed!")

encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = decoder_embedding_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2)

reverse_target_word_index = ml_tokenizer.index_word
start_token_id = ml_tokenizer.word_index['<start>']
end_token_id = ml_tokenizer.word_index['<end>']

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = start_token_id

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence.split()) > 20:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word
            target_seq = np.zeros((1, 1))
            target_seq[0, 0] = sampled_token_index
            states_value = [h, c]

    return decoded_sentence.strip()

test_seq = en_tokenizer.texts_to_sequences(["I am reading"])
test_input = pad_sequences(test_seq, maxlen=encoder_input_data.shape[1], padding='post')

translated = decode_sequence(test_input)
print("Translated Output:", translated)

Training Completed!


Translated Output: ഞാൻ വായിക്കുന്നു


TESTING WITH ANOTHER WORD

In [29]:
## TEST
test_seq = en_tokenizer.texts_to_sequences(["I am coming"])
test_input = pad_sequences(test_seq, maxlen=encoder_input_data.shape[1], padding='post')

translated = decode_sequence(test_input)
print("Translated Output:", translated)

Translated Output: ഞാൻ വരുന്നു
